# Sulphur-2-base Video Generation (AMD Radeon Cloud)

**Model:** [SulphurAI/Sulphur-2-base](https://huggingface.co/SulphurAI/Sulphur-2-base)  
**Architecture:** DiT-based (LTX 2.3) | ~9B params  
**Platform:** AMD Radeon Cloud (ROCm) + OneDrive  

---

### 工作原理

1. 通过 rclone 挂载你的 OneDrive（1TB 空间足够存模型和视频）
2. 模型存储在 OneDrive 中，实例重启不丢失
3. 首次需要在本地电脑下载模型并上传到 OneDrive（一次性），之后每次启动只需挂载（几秒钟）

### 网络限制说明

AMD Radeon Cloud 实例**无法访问** HuggingFace 和 Google APIs，  
但**可以访问** OneDrive (graph.microsoft.com)。  
因此模型需要先在本地下载，再通过 OneDrive 同步到云端。

---
## 1. 挂载 OneDrive

**每次启动实例后运行这个 cell**，几秒钟完成挂载。

- 首次使用？先跳到下方 "首次配置" 部分完成授权
- 已经配置过？直接运行，自动挂载

In [ ]:
# ============================================================
# 配置区域（修改这里）
# ============================================================

# 挂载点
MOUNT_POINT = "/mnt/my-drive"

# OneDrive 内的目录规划（所有资产都存这里，重启不丢）
MODEL_DIR   = f"{MOUNT_POINT}/models/Sulphur-2-base"   # 模型
OUTPUT_DIR  = f"{MOUNT_POINT}/outputs/sulphur2"        # 生成的视频
ASSETS_DIR  = f"{MOUNT_POINT}/assets"                  # 参考图片等素材
WORKSPACE   = f"{MOUNT_POINT}/workspace"               # 其他工作文件

# rclone 配置（首次配置后粘贴到这里，之后每次启动自动生效）
# 获取方式：在本地电脑运行 rclone authorize "onedrive" 后生成
RCLONE_CONFIG = """
[my-drive]
type = onedrive
drive_type = personal
token = {"access_token":"YOUR_TOKEN_HERE","token_type":"Bearer","refresh_token":"YOUR_REFRESH_TOKEN","expiry":"2026-01-01T00:00:00+08:00"}
drive_id = YOUR_DRIVE_ID
""".strip()

In [ ]:
import subprocess, os
from pathlib import Path

# --- 安装系统依赖 ---
subprocess.run("apt-get update -qq && apt-get install -y -qq unzip fuse", shell=True, capture_output=True)

# --- 安装 rclone ---
if subprocess.run(["which", "rclone"], capture_output=True).returncode != 0:
    print("Installing rclone...")
    subprocess.run("curl -s https://rclone.org/install.sh | bash", shell=True, check=True)
else:
    print("rclone already installed.")

# --- 写入配置 ---
rclone_conf_dir = Path.home() / ".config" / "rclone"
rclone_conf_dir.mkdir(parents=True, exist_ok=True)
rclone_conf_path = rclone_conf_dir / "rclone.conf"

if RCLONE_CONFIG and "YOUR_TOKEN_HERE" not in RCLONE_CONFIG:
    rclone_conf_path.write_text(RCLONE_CONFIG)
    print(f"Config written to {rclone_conf_path}")
else:
    print("WARNING: RCLONE_CONFIG not set! Run the '首次配置' cell below first.")

# --- 挂载云盘 ---
mount_path = Path(MOUNT_POINT)
mount_path.mkdir(parents=True, exist_ok=True)

# 检查是否已挂载
if any(mount_path.iterdir()) if mount_path.exists() else False:
    print(f"Cloud drive already mounted at {MOUNT_POINT}")
else:
    print(f"Mounting cloud drive to {MOUNT_POINT}...")
    subprocess.Popen([
        "rclone", "mount", "my-drive:/", MOUNT_POINT,
        "--vfs-cache-mode", "full",
        "--vfs-cache-max-size", "30G",
        "--vfs-read-chunk-size", "64M",
        "--vfs-read-chunk-size-limit", "512M",
        "--buffer-size", "128M",
        "--daemon",
    ])
    import time
    time.sleep(3)
    if any(mount_path.iterdir()):
        print(f"Mounted successfully!")
    else:
        print("ERROR: Mount failed. Check rclone config.")

# --- 检查模型是否已存在 ---
model_path = Path(MODEL_DIR)
if model_path.exists() and any(model_path.glob("*.safetensors")) or any(model_path.glob("**/*.safetensors")):
    print(f"\nModel found at: {MODEL_DIR}")
    print("No download needed! Ready to load.")
else:
    print(f"\nModel NOT found at: {MODEL_DIR}")
    print("Run the '首次下载模型' cell below to download it to your cloud drive.")

### 首次配置（只需做一次）

OneDrive 授权需要在**你的本地电脑**上完成（因为实例没有浏览器）。

#### 步骤：

1. 在本地电脑下载 rclone: https://rclone.org/downloads/
2. 打开命令行，设置代理（如果需要）：
   ```
   set HTTP_PROXY=http://127.0.0.1:7890
   set HTTPS_PROXY=http://127.0.0.1:7890
   ```
3. 运行授权命令：
   ```
   rclone authorize "onedrive"
   ```
4. 浏览器会自动打开，登录你的 Microsoft 账号并授权
5. 授权成功后，命令行会输出 token
6. 然后运行完整配置命令：
   ```
   rclone config create my-drive onedrive drive_type personal
   ```
7. 查看配置：
   ```
   rclone config dump
   ```
8. 把输出的配置（包含 token 和 drive_id）粘贴到上方 `RCLONE_CONFIG` 变量中

配置完成后，以后每次启动实例只需运行上方的挂载 cell 即可。

In [ ]:
# 首次配置辅助 cell（在实例上验证连接是否正常）
import subprocess
from pathlib import Path

# 安装系统依赖 + rclone
subprocess.run("apt-get update -qq && apt-get install -y -qq unzip fuse", shell=True, capture_output=True)
if subprocess.run(["which", "rclone"], capture_output=True).returncode != 0:
    print("Installing rclone...")
    subprocess.run("curl -s https://rclone.org/install.sh | bash", shell=True, check=True)
else:
    print("rclone already installed.")

# 写入配置并测试连接
rclone_conf_dir = Path.home() / ".config" / "rclone"
rclone_conf_dir.mkdir(parents=True, exist_ok=True)
rclone_conf_path = rclone_conf_dir / "rclone.conf"

if RCLONE_CONFIG and "YOUR_TOKEN_HERE" not in RCLONE_CONFIG:
    rclone_conf_path.write_text(RCLONE_CONFIG)
    print(f"Config written to {rclone_conf_path}")
    print("\nTesting connection...")
    result = subprocess.run(["rclone", "lsd", "my-drive:/"], capture_output=True, text=True, timeout=30)
    if result.returncode == 0:
        print("Connection successful! Your OneDrive folders:")
        print(result.stdout)
    else:
        print(f"ERROR: {result.stderr}")
else:
    print("请先在上方 RCLONE_CONFIG 中填入你的 OneDrive 配置！")
    print("参考上方 '首次配置' 说明获取配置。")

### 首次上传模型到 OneDrive（只需做一次）

由于 Radeon Cloud 实例**无法访问 HuggingFace**，模型需要在本地电脑下载后上传到 OneDrive。

#### 在本地电脑操作：

```bash
# 1. 安装 huggingface-hub（如果没有）
pip install huggingface-hub

# 2. 下载模型到本地（约 20GB）
huggingface-cli download SulphurAI/Sulphur-2-base --local-dir ./Sulphur-2-base

# 3. 用 rclone 上传到 OneDrive（本地已配置好 rclone 的情况下）
rclone copy ./Sulphur-2-base my-drive:/models/Sulphur-2-base --progress

# 或者直接把 Sulphur-2-base 文件夹拖到 OneDrive 的 models 目录下（通过 OneDrive 客户端同步）
```

上传完成后，在实例上挂载 OneDrive 就能直接读取模型，不用再下载。

In [ ]:
# 验证模型是否已上传到 OneDrive（在实例上运行）
import subprocess

print("Checking if model exists on OneDrive...")
result = subprocess.run(
    ["rclone", "ls", "my-drive:/models/Sulphur-2-base", "--max-depth", "1"],
    capture_output=True, text=True, timeout=30
)

if result.returncode == 0 and result.stdout.strip():
    print("Model found on OneDrive!")
    print("Files:")
    for line in result.stdout.strip().split("\n")[:10]:
        print(f"  {line}")
    total = len(result.stdout.strip().split("\n"))
    if total > 10:
        print(f"  ... and {total - 10} more files")
else:
    print("Model NOT found on OneDrive.")
    print("请在本地电脑下载模型并上传到 OneDrive:/models/Sulphur-2-base")
    print("参考上方说明操作。")

---
## 2. 安装依赖 + 加载模型

In [ ]:
%%time
import subprocess, sys

# 安装 PyTorch（优先使用容器预装版本，否则安装 ROCm 版）
try:
    import torch
    assert torch.cuda.is_available()
    print(f"PyTorch already installed: {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}")
except (ImportError, AssertionError):
    print("Installing PyTorch for ROCm...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "torch", "torchvision", "torchaudio",
        "--index-url", "https://download.pytorch.org/whl/rocm6.2"], check=False)
    import torch
    print(f"PyTorch: {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}")

# 安装其他依赖
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "diffusers", "transformers", "accelerate", "safetensors",
    "sentencepiece", "imageio[ffmpeg]", "pillow", "huggingface_hub"], check=True)

from diffusers import LTXPipeline

# 从云盘加载模型（不下载，直接读取）
print(f"\nLoading model from cloud drive: {MODEL_DIR}")
pipe = LTXPipeline.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.bfloat16,
)
pipe.to("cuda")
print("Model loaded!")

---
## 3. Text-to-Video 生成视频

In [ ]:
# ============================================================
# 修改 prompt 生成你想要的视频
# ============================================================
prompt = (
    "A cinematic shot of a golden retriever running through a sunlit meadow, "
    "slow motion, shallow depth of field, warm color grading, 4K quality"
)

negative_prompt = "blurry, low quality, distorted, watermark, text"

video = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_frames=97,              # 49(短) / 97(中) / 161(长)
    width=768,
    height=512,
    num_inference_steps=50,     # 30(快) / 50(标准)
    guidance_scale=7.5,
    generator=torch.Generator("cuda").manual_seed(42),
).frames[0]

print(f"Generated {len(video)} frames")

## 4. 保存视频到云盘 + 预览

视频直接保存到云盘，重启也不丢失！

In [ ]:
from diffusers.utils import export_to_video
from IPython.display import Video, display, FileLink
from pathlib import Path
from datetime import datetime

# 保存到云盘的 outputs 目录（持久化！）
output_dir = Path(MOUNT_POINT) / "outputs" / "sulphur2"
output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = output_dir / f"t2v_{timestamp}.mp4"

export_to_video(video, str(output_path), fps=24)

print(f"Video saved to cloud drive: {output_path}")
print(f"Size: {output_path.stat().st_size / 1024 / 1024:.1f} MB")
print("This file persists in your cloud drive even after restart!")
print("\n--- Preview: ---")
display(Video(str(output_path), embed=True, width=640))

---
## 5. Image-to-Video（可选）

In [ ]:
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import load_image

IMAGE_PATH = "your_image.png"  # 修改为你的图片路径

i2v_pipe = LTXImageToVideoPipeline.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.bfloat16,
)
i2v_pipe.to("cuda")

image = load_image(IMAGE_PATH).resize((768, 512))

video_i2v = i2v_pipe(
    prompt="The scene comes to life with gentle motion, cinematic lighting",
    image=image,
    num_frames=97,
    width=768,
    height=512,
    num_inference_steps=50,
    guidance_scale=7.5,
    generator=torch.Generator("cuda").manual_seed(42),
).frames[0]

i2v_path = output_dir / f"i2v_{timestamp}.mp4"
export_to_video(video_i2v, str(i2v_path), fps=24)
print(f"Saved to cloud drive: {i2v_path}")
display(Video(str(i2v_path), embed=True, width=640))

---
## 参数参考

| 参数 | 推荐值 | 说明 |
|------|--------|------|
| `num_frames` | 49 / 97 / 161 | 帧数 |
| `width` x `height` | 768x512 / 1024x576 | 分辨率 |
| `num_inference_steps` | 30-50 | 质量 vs 速度 |
| `guidance_scale` | 5.0-9.0 | prompt 贴合度 |
| `seed` | 任意整数 | 可复现结果 |

## 每次启动的流程

```
启动实例 → 运行 Cell 1（挂载 OneDrive，几秒）→ 运行 Cell 2（装依赖+加载模型）→ 生成视频
```

模型和生成的视频都在 OneDrive 里，永不丢失。

## 注意事项

- 实例无法访问 HuggingFace，模型必须通过 OneDrive 传入
- rclone token 有效期约 1 小时，但 refresh_token 会自动续期
- 如果 token 过期报错，在本地重新运行 `rclone authorize "onedrive"` 更新配置